<!-- project-navigation -->
# decision_tree
Role: Model experiments and analysis.

See [notebook guide](README.md) and [model comparison](../reports/model_comparison.md). Historical code and outputs are retained; run with the project root as working directory.


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

# ============================================================
# CHARGEMENT DES DONNÉES
# ============================================================

df = pd.read_csv("data/processed/train_encoded.csv")

print("Dimension :", df.shape)

# Séparation cible / variables
X = df.drop(columns=["TARGET", "SK_ID_CURR"], errors="ignore")
y = df["TARGET"]

print("X :", X.shape)
print("y :", y.shape)

print("\nDistribution TARGET :")
print(y.value_counts())
print(y.value_counts(normalize=True))

Dimension : (307511, 242)
X : (307511, 240)
y : (307511,)

Distribution TARGET :
TARGET
0    282686
1     24825
Name: count, dtype: int64
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [3]:
from sklearn.model_selection import train_test_split

# ============================================================
# TRAIN / VALIDATION SPLIT
# ============================================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("========== DIMENSIONS ==========")
print("X_train :", X_train.shape)
print("X_valid :", X_valid.shape)
print("y_train :", y_train.shape)
print("y_valid :", y_valid.shape)

print("\n========== DISTRIBUTION TRAIN ==========")
print(y_train.value_counts(normalize=True))

print("\n========== DISTRIBUTION VALIDATION ==========")
print(y_valid.value_counts(normalize=True))

========== DIMENSIONS ==========
X_train : (246008, 240)
X_valid : (61503, 240)
y_train : (246008,)
y_valid : (61503,)

========== DISTRIBUTION TRAIN ==========
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

========== DISTRIBUTION VALIDATION ==========
TARGET
0    0.919272
1    0.080728
Name: proportion, dtype: float64


In [4]:
from sklearn.tree import DecisionTreeClassifier

# ============================================================
# DECISION TREE — BASELINE
# ============================================================

model_dt = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"
)

model_dt.fit(X_train, y_train)

print("Decision Tree entraîné.")

Decision Tree entraîné.


In [5]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# ÉVALUATION DECISION TREE
# ============================================================

y_pred_dt = model_dt.predict(X_valid)
y_proba_dt = model_dt.predict_proba(X_valid)[:, 1]

print("========== DECISION TREE — BASELINE ==========")

print(f"Accuracy      : {accuracy_score(y_valid, y_pred_dt):.4f}")
print(f"Precision     : {precision_score(y_valid, y_pred_dt):.4f}")
print(f"Recall défaut : {recall_score(y_valid, y_pred_dt):.4f}")
print(f"F1-score      : {f1_score(y_valid, y_pred_dt):.4f}")
print(f"ROC-AUC       : {roc_auc_score(y_valid, y_proba_dt):.4f}")

print("\n========== MATRICE DE CONFUSION ==========")
print(confusion_matrix(y_valid, y_pred_dt))

print("\n========== CLASSIFICATION REPORT ==========")
print(classification_report(y_valid, y_pred_dt))

========== DECISION TREE — BASELINE ==========
Accuracy      : 0.8608
Precision     : 0.1601
Recall défaut : 0.1706
F1-score      : 0.1652
ROC-AUC       : 0.5460

========== MATRICE DE CONFUSION ==========
[[52096  4442]
 [ 4118   847]]

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

           0       0.93      0.92      0.92     56538
           1       0.16      0.17      0.17      4965

    accuracy                           0.86     61503
   macro avg       0.54      0.55      0.54     61503
weighted avg       0.86      0.86      0.86     61503



In [6]:
# ============================================================
# HYPERPARAMÈTRES — DECISION TREE BASELINE
# ============================================================

print("========== HYPERPARAMÈTRES DECISION TREE ==========")

params_dt = model_dt.get_params()

for param, value in params_dt.items():
    print(f"{param} : {value}")

========== HYPERPARAMÈTRES DECISION TREE ==========
ccp_alpha : 0.0
class_weight : balanced
criterion : gini
max_depth : None
max_features : None
max_leaf_nodes : None
min_impurity_decrease : 0.0
min_samples_leaf : 1
min_samples_split : 2
min_weight_fraction_leaf : 0.0
monotonic_cst : None
random_state : 42
splitter : best


In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV

# ============================================================
# DECISION TREE — OPTIMISATION DES HYPERPARAMÈTRES
# ============================================================

dt = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"
)

param_distributions_dt = {
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": [5, 10, 15, 20, 30, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": [None, "sqrt", "log2"],
    "ccp_alpha": [0.0, 0.001, 0.005, 0.01]
}

random_search_dt = RandomizedSearchCV(
    estimator=dt,
    param_distributions=param_distributions_dt,
    n_iter=10,
    scoring="roc_auc",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=1
)

print("==========================================")
print("DÉBUT RANDOMIZED SEARCH — DECISION TREE")
print("==========================================")

random_search_dt.fit(X_train, y_train)

print("\n==========================================")
print("RECHERCHE TERMINÉE")
print("==========================================")

print("\nMeilleurs paramètres :")
print(random_search_dt.best_params_)

print("\nMeilleur ROC-AUC CV :")
print(random_search_dt.best_score_)

DÉBUT RANDOMIZED SEARCH — DECISION TREE
Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END ccp_alpha=0.01, criterion=log_loss, max_depth=5, max_features=None, min_samples_leaf=2, min_samples_split=10; total time=  14.9s
[CV] END ccp_alpha=0.01, criterion=log_loss, max_depth=5, max_features=None, min_samples_leaf=2, min_samples_split=10; total time=  13.4s
[CV] END ccp_alpha=0.01, criterion=log_loss, max_depth=5, max_features=None, min_samples_leaf=2, min_samples_split=10; total time=  13.2s
[CV] END ccp_alpha=0.0, criterion=log_loss, max_depth=None, max_features=log2, min_samples_leaf=10, min_samples_split=2; total time=   2.6s
[CV] END ccp_alpha=0.0, criterion=log_loss, max_depth=None, max_features=log2, min_samples_leaf=10, min_samples_split=2; total time=   2.4s
[CV] END ccp_alpha=0.0, criterion=log_loss, max_depth=None, max_features=log2, min_samples_leaf=10, min_samples_split=2; total time=   2.6s
[CV] END ccp_alpha=0.001, criterion=entropy, max_depth=15, max_fe

In [8]:
# ============================================================
# DECISION TREE OPTIMISÉ — ÉVALUATION VALIDATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Meilleur modèle trouvé
best_dt = random_search_dt.best_estimator_

# Prédictions
y_pred_dt_opt = best_dt.predict(X_valid)
y_proba_dt_opt = best_dt.predict_proba(X_valid)[:, 1]

# Métriques
accuracy_dt_opt = accuracy_score(y_valid, y_pred_dt_opt)
precision_dt_opt = precision_score(y_valid, y_pred_dt_opt)
recall_dt_opt = recall_score(y_valid, y_pred_dt_opt)
f1_dt_opt = f1_score(y_valid, y_pred_dt_opt)
roc_auc_dt_opt = roc_auc_score(y_valid, y_proba_dt_opt)

print("========== DECISION TREE — OPTIMISÉ ==========")
print(f"Accuracy      : {accuracy_dt_opt:.4f}")
print(f"Precision     : {precision_dt_opt:.4f}")
print(f"Recall défaut : {recall_dt_opt:.4f}")
print(f"F1-score      : {f1_dt_opt:.4f}")
print(f"ROC-AUC       : {roc_auc_dt_opt:.4f}")

print("\n========== MATRICE DE CONFUSION ==========")
print(confusion_matrix(y_valid, y_pred_dt_opt))

print("\n========== CLASSIFICATION REPORT ==========")
print(classification_report(y_valid, y_pred_dt_opt))

========== DECISION TREE — OPTIMISÉ ==========
Accuracy      : 0.6086
Precision     : 0.1349
Recall défaut : 0.7108
F1-score      : 0.2267
ROC-AUC       : 0.7042

========== MATRICE DE CONFUSION ==========
[[33899 22639]
 [ 1436  3529]]

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

           0       0.96      0.60      0.74     56538
           1       0.13      0.71      0.23      4965

    accuracy                           0.61     61503
   macro avg       0.55      0.66      0.48     61503
weighted avg       0.89      0.61      0.70     61503

